### Imports

In [ ]:
from qarp.blocks import TrotterBlock, TrotterAnsatzBlock
from qarp.operators import JordanWigner
from qarp.operators.models import fermi_hubbard
from qarp.operators.ucc import ucc_singles_and_doubles
from qarp.plotting import plot

from qarp.operators.functions import count_qubits
from qarp.operators import FullyCommuting, NoGrouping
from sympy import Symbol
import numpy as np

### Trotter object

In [ ]:
# Hamiltonian Trotterization

# Trotterization parameters
time = 1
n_trotter_steps = 1
trotter_order = 2
grouping = FullyCommuting()

# Hamiltonian
fham = fermi_hubbard((2,), t=0.14, U=0.231)
qham = JordanWigner().encode_operator(fham)
n_qubits = count_qubits(qham)

# Trotter circuit generated by the Hamiltonian
circ = TrotterBlock(operator=qham, n_qubits=n_qubits, steps=n_trotter_steps, time=time, order=trotter_order, grouping=grouping).build()

In [ ]:
circ.plot(scrollable=True)

### Symbolic Time

You can use a symbolic time parameter to build the circuit once and then vary the time without rebuilding. Use `set_time()` to substitute time values.

In [ ]:
# Create TrotterBlock with symbolic time
t = Symbol('t')

trotter_symbolic = TrotterBlock(
    operator=qham,
    n_qubits=n_qubits,
    steps=n_trotter_steps,
    time=t,  # Symbolic time parameter
    order=trotter_order,
    grouping=grouping
)
trotter_symbolic.build()

print(f"Circuit has symbolic time: {trotter_symbolic.symbols}")

# Vary time without rebuilding using set_time()
trotter_t05 = trotter_symbolic.set_time(0.5)
trotter_t10 = trotter_symbolic.set_time(1.0)
trotter_t20 = trotter_symbolic.set_time(2.0)

# Plot circuits for different times
trotter_t05.plot(scrollable=True)
trotter_t10.plot(scrollable=True)
trotter_t20.plot(scrollable=True)

print(f"After set_time(0.5): {trotter_t05.symbols}")
print(f"After set_time(1.0): {trotter_t10.symbols}")
print(f"After set_time(2.0): {trotter_t20.symbols}")

### TrotterAnsatzBlock

`TrotterAnsatzBlock` is used for variational quantum algorithms where each operator term has its own variational parameter (symbol). This is commonly used in VQE with UCC ansätze.

In [ ]:
# TrotterAnsatzBlock with UCC excitations
# Each operator gets its own variational parameter

onv = [1, 1, 0, 0]  # 2 electrons in 4 spin-orbitals
fermionic_excitations, ansatz_symbols = ucc_singles_and_doubles(onv)
qubit_excitations = JordanWigner().encode_operator(fermionic_excitations)

ansatz = TrotterAnsatzBlock(
    n_qubits=4,
    qubit_exponents=qubit_excitations,
    symbols=ansatz_symbols,
    steps=1,
    time=1.0,
    order=1,
    grouping=NoGrouping(),
    imaginary=True,
)
ansatz.build()

print(f"Number of variational parameters: {len(ansatz.symbols)}")
print(f"Symbols: {ansatz.symbols}")
ansatz.plot(scrollable=True)

### TrotterAnsatzBlock with Symbolic Time

You can also combine variational parameters with symbolic time in `TrotterAnsatzBlock`.

In [ ]:
# TrotterAnsatzBlock with symbolic time
t_ansatz = Symbol('t')

ansatz_symbolic = TrotterAnsatzBlock(
    n_qubits=4,
    qubit_exponents=qubit_excitations,
    symbols=ansatz_symbols.copy(),
    steps=1,
    time=t_ansatz,  # Symbolic time
    order=1,
    grouping=NoGrouping(),
    imaginary=True,
)
ansatz_symbolic.build()

print(f"All symbols (including time): {ansatz_symbolic.symbols}")
print(f"Time symbol tracked: {ansatz_symbolic._time_symbol}")
print(f"Circuit free symbols: {ansatz_symbolic.symbols}")

# Substitute time while keeping variational parameters symbolic
ansatz_t05 = ansatz_symbolic.set_time(0.5)
print(f"\nAfter set_time(0.5), remaining symbols: {ansatz_t05.symbols}")

### Setting Variational Parameters

Use `set_symbols()` to substitute variational parameter values.

In [ ]:
# Substitute all parameters (ansatz parameters + time)
# First set time, then set the variational parameters
ansatz_with_time = ansatz_symbolic.set_time(1.0)

# Create parameter values for the ansatz symbols
param_values = {s: 0.1 * (i + 1) for i, s in enumerate(ansatz_symbols)}
print(f"Parameter values: {param_values}")

# Substitute the variational parameters
ansatz_final = ansatz_with_time.set_symbols(param_values)
print(f"Remaining free symbols: {ansatz_final.symbols}")